# 03 — IT2-ANFIS training

Train Interval Type-2 ANFIS (e.g. 7 rules), plot true vs predicted with R², then visualize trained membership functions (upper/lower bounds).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_csv, clean_and_prepare
from src.features import select_features_mi
from src.models.it2_anfis import IT2_TSK_ANFIS
from src.experiments.train import run_it2_training
from src.viz import plot_membership_functions
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

path = Path("../data/raw/Data-Melbourne_F_fixed.csv")
if not path.exists():
    path = Path("../../Data-Melbourne_F_fixed.csv")
df = load_csv(path)
df_clean = clean_and_prepare(df)
df_final = select_features_mi(df_clean)
X = df_final.drop(columns=['Energy Consumption']).values
y = df_final['Energy Consumption'].values
feature_names = df_final.drop(columns=['Energy Consumption']).columns.tolist()

In [ ]:
model, (X_tr, X_val, X_test, y_tr, y_val, y_test) = run_it2_training(
    X, y, n_rules=7, epochs=80, patience=25, verbose=True
)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"Test R² = {r2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred, alpha=0.5, s=20)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('True values')
ax.set_ylabel('Predicted')
ax.set_title(f'True vs Predicted (R² = {r2:.4f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_membership_functions(model, feature_names=feature_names, selected_rules=[0, 2, 4, 6])
plt.show()